# Phase A: Fine-tune FunctionGemma 270M IT

This notebook fine-tunes `google/functiongemma-270m-it` on your custom function calling dataset.

**Output**: A HuggingFace-format model directory (SafeTensors) ready for GGUF conversion.

**Reference**: Based on [flutter_gemma finetuning notebook](https://github.com/DenisovAV/flutter_gemma/blob/main/colabs/functiongemma_finetuning.ipynb)

**Runtime**: Select GPU (T4 or better) from Runtime > Change runtime type

## Step 1: Install Dependencies

In [ ]:
!pip install -q \
    transformers>=4.50.0 \
    trl>=0.26.0 \
    datasets \
    accelerate \
    torch \
    huggingface_hub

## Step 2: Login to HuggingFace

FunctionGemma requires accepting the Gemma license on HuggingFace.

1. Go to https://huggingface.co/google/functiongemma-270m-it
2. Accept the license agreement
3. Create a token at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login

# Option 1: Interactive login
login()

# Option 2: Direct token (uncomment and paste your token)
# login(token="hf_YOUR_TOKEN_HERE")

## Step 3: Load Base Model and Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "google/functiongemma-270m-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")

## Step 4: Prepare Training Data

### Option A: Upload your own JSONL file

Upload a file named `training_data.jsonl` with this format:
```json
{"user_content": "Turn on the light", "tool_name": "turn_on_light", "tool_arguments": {"room": "living room"}}
```

### Option B: Use the sample data below

In [ ]:
import json
import os

# ===== CONFIGURATION =====
# Set to True to use uploaded file, False to use sample data
USE_UPLOADED_DATA = False
UPLOADED_FILE_PATH = "training_data.jsonl"  # Upload this to Colab

# ===== Define your functions =====
# These are the functions your model will learn to call.
# Modify this list to match your use case.
FUNCTION_DECLARATIONS = [
    {
        "name": "turn_on_light",
        "description": "Turn on a light in a specific room",
        "parameters": {
            "type": "object",
            "properties": {
                "room": {"type": "string", "description": "The room name"}
            },
            "required": ["room"]
        }
    },
    {
        "name": "turn_off_light",
        "description": "Turn off a light in a specific room",
        "parameters": {
            "type": "object",
            "properties": {
                "room": {"type": "string", "description": "The room name"}
            },
            "required": ["room"]
        }
    },
    {
        "name": "set_temperature",
        "description": "Set the thermostat temperature",
        "parameters": {
            "type": "object",
            "properties": {
                "temperature": {"type": "number", "description": "Target temperature"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["temperature"]
        }
    },
    {
        "name": "get_temperature",
        "description": "Get the current temperature in a room",
        "parameters": {
            "type": "object",
            "properties": {
                "room": {"type": "string", "description": "The room name"}
            },
            "required": ["room"]
        }
    },
]

# ===== Sample training data (modify or replace) =====
SAMPLE_DATA = [
    {"user_content": "Turn on the living room light", "tool_name": "turn_on_light", "tool_arguments": {"room": "living room"}},
    {"user_content": "Turn on the kitchen light", "tool_name": "turn_on_light", "tool_arguments": {"room": "kitchen"}},
    {"user_content": "Turn on the bedroom light", "tool_name": "turn_on_light", "tool_arguments": {"room": "bedroom"}},
    {"user_content": "Turn on the bathroom light", "tool_name": "turn_on_light", "tool_arguments": {"room": "bathroom"}},
    {"user_content": "Please switch on the light in the hallway", "tool_name": "turn_on_light", "tool_arguments": {"room": "hallway"}},
    {"user_content": "Turn off the living room light", "tool_name": "turn_off_light", "tool_arguments": {"room": "living room"}},
    {"user_content": "Turn off the kitchen light", "tool_name": "turn_off_light", "tool_arguments": {"room": "kitchen"}},
    {"user_content": "Switch off the bedroom light", "tool_name": "turn_off_light", "tool_arguments": {"room": "bedroom"}},
    {"user_content": "Set the thermostat to 24 degrees", "tool_name": "set_temperature", "tool_arguments": {"temperature": 24, "unit": "celsius"}},
    {"user_content": "Set temperature to 22 celsius", "tool_name": "set_temperature", "tool_arguments": {"temperature": 22, "unit": "celsius"}},
    {"user_content": "Make it 20 degrees in here", "tool_name": "set_temperature", "tool_arguments": {"temperature": 20, "unit": "celsius"}},
    {"user_content": "Set the temperature to 75 fahrenheit", "tool_name": "set_temperature", "tool_arguments": {"temperature": 75, "unit": "fahrenheit"}},
    {"user_content": "What is the temperature in the kitchen?", "tool_name": "get_temperature", "tool_arguments": {"room": "kitchen"}},
    {"user_content": "How warm is the living room?", "tool_name": "get_temperature", "tool_arguments": {"room": "living room"}},
    {"user_content": "Check the bedroom temperature", "tool_name": "get_temperature", "tool_arguments": {"room": "bedroom"}},
    {"user_content": "What's the temp in the office?", "tool_name": "get_temperature", "tool_arguments": {"room": "office"}},
]

print(f"Sample data: {len(SAMPLE_DATA)} examples")
print(f"Functions: {[f['name'] for f in FUNCTION_DECLARATIONS]}")

## Step 5: Format Data for FunctionGemma

**IMPORTANT**: FunctionGemma uses a custom chat format with special tokens.
Do NOT use `apply_chat_template` - it produces a different format.

In [ ]:
from datasets import Dataset


def format_function_declarations(functions):
    """Format function declarations using FunctionGemma special tokens."""
    result = ""
    for f in functions:
        result += f"\n<start_function_declaration>\n{json.dumps(f)}\n<end_function_declaration>\n"
    return result


def format_training_example(example, functions):
    """Format a single training example into FunctionGemma format."""
    func_decl = format_function_declarations(functions)

    function_call = json.dumps({
        "name": example["tool_name"],
        "arguments": example["tool_arguments"]
    })

    text = (
        f"<start_of_turn>user\n"
        f"You are a helpful assistant with access to the following functions.\n"
        f"Use them if required:\n"
        f"{func_decl}\n"
        f"{example['user_content']}\n"
        f"<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"<start_function_call>\n"
        f"{function_call}\n"
        f"<end_function_call>\n"
        f"<end_of_turn>"
    )
    return text


# Load data
if USE_UPLOADED_DATA and os.path.exists(UPLOADED_FILE_PATH):
    with open(UPLOADED_FILE_PATH, 'r') as f:
        raw_data = [json.loads(line) for line in f if line.strip()]
    print(f"Loaded {len(raw_data)} examples from {UPLOADED_FILE_PATH}")
else:
    raw_data = SAMPLE_DATA
    print(f"Using {len(raw_data)} sample examples")

# Format all examples
formatted_texts = [
    format_training_example(ex, FUNCTION_DECLARATIONS)
    for ex in raw_data
]

# Create HuggingFace dataset
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"\nDataset size: {len(dataset)}")
print(f"\n--- Example formatted text ---")
print(formatted_texts[0][:500])
print("...")

## Step 6: Configure Training

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./finetuned_model"

# Training configuration
# Adjust these based on your dataset size and available GPU
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    
    # Training hyperparameters
    num_train_epochs=5,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    
    # Batch size
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,  # Effective batch size = 4 * 8 = 32
    
    # Sequence length
    max_seq_length=1024,
    
    # Precision
    bf16=True,
    
    # Logging
    logging_steps=5,
    save_strategy="epoch",
    
    # Dataset
    dataset_text_field="text",
    packing=False,
)

print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Max seq length: {training_args.max_seq_length}")

## Step 7: Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

print("Starting training...")
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.1f}s")

## Step 8: Save Model

In [ ]:
# Save the fine-tuned model and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Verify output files
print("Saved files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024 * 1024)
    print(f"  {f} ({size_mb:.1f} MB)")

## Step 9: Test the Fine-tuned Model

In [ ]:
# Reload the saved model for testing
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Test queries
test_queries = [
    "Turn on the garage light",
    "Set temperature to 26 degrees",
    "What's the temperature in the office?",
]

func_decl = format_function_declarations(FUNCTION_DECLARATIONS)

for query in test_queries:
    prompt = (
        f"<start_of_turn>user\n"
        f"You are a helpful assistant with access to the following functions.\n"
        f"Use them if required:\n"
        f"{func_decl}\n"
        f"{query}\n"
        f"<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

    output = pipe(
        prompt,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=True,
    )

    generated = output[0]["generated_text"][len(prompt):]
    print(f"Query: {query}")
    print(f"Output: {generated.strip()}")
    print()

## Step 10: Save to Google Drive (for GGUF conversion)

Save the model to Google Drive so you can access it in the next notebook (02_convert_to_gguf.ipynb).

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Copy to Google Drive
DRIVE_OUTPUT = "/content/drive/MyDrive/functiongemma_finetuned"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy all model files
for f in os.listdir(OUTPUT_DIR):
    src = os.path.join(OUTPUT_DIR, f)
    dst = os.path.join(DRIVE_OUTPUT, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        print(f"Copied: {f}")

print(f"\nModel saved to Google Drive: {DRIVE_OUTPUT}")
print("You can now proceed to 02_convert_to_gguf.ipynb")